In [1]:
# Cell 1 — Imports
import onnxruntime as rt
import numpy as np, pandas as pd, json, time, os, joblib

MODELS = 'models/'
DATA   = 'data/processed/'

# Cell 2 — Load ONNX session + scaler
sess    = rt.InferenceSession(os.path.join(MODELS, 'xgb_edge.onnx'))
scaler  = joblib.load(os.path.join(MODELS, 'scaler_unified_4dataset.pkl'))
in_name = sess.get_inputs()[0].name

# Cell 3 — Load a test slice (TON-IoT as worst-case)
df = pd.read_parquet(os.path.join(DATA, 'ton_iot_test.parquet'))
SFAF_FEATURES = [...]  # paste your 12 feature names here
X_test = scaler.transform(df[SFAF_FEATURES].values).astype(np.float32)
y_test = df['label'].values

# Cell 4 — Latency benchmark (simulate Raspberry Pi 4 thread constraint)
BATCH_SIZES = [1, 8, 32, 64, 128, 512]
results = []
for bs in BATCH_SIZES:
    batch = X_test[:bs]
    times = []
    for _ in range(200):          # 200 warmup+measure runs
        t0 = time.perf_counter()
        sess.run(None, {in_name: batch})
        times.append((time.perf_counter() - t0) * 1000)   # ms
    times = times[10:]            # drop warmup
    results.append({
        'batch_size': bs,
        'mean_ms':    round(np.mean(times), 3),
        'p99_ms':     round(np.percentile(times, 99), 3),
        'throughput': round(bs / (np.mean(times) / 1000), 1)
    })

df_lat = pd.DataFrame(results)
df_lat.to_csv(os.path.join(MODELS, 'latency_by_batch.csv'), index=False)
print(df_lat.to_string(index=False))

# Cell 5 — Accuracy on test set via ONNX (verify parity with pkl model)
from sklearn.metrics import accuracy_score, f1_score
preds = sess.run(None, {in_name: X_test})[0]
print(f"ONNX Accuracy: {accuracy_score(y_test, preds):.4f}")
print(f"ONNX F1:       {f1_score(y_test, preds):.4f}")

# Cell 6 — Model size comparison table (for report)
sizes = {
    'xgb_unified_4dataset.pkl': 405,   # KB
    'xgb_edge.pkl':              37,
    'xgb_edge.onnx':             46,
}
df_size = pd.DataFrame(sizes.items(), columns=['Model', 'Size_KB'])
df_size['Reduction_%'] = round((1 - df_size.Size_KB / 405) * 100, 1)
df_size.to_csv(os.path.join(MODELS, 'model_size_comparison.csv'), index=False)
print(df_size)


NoSuchFile: [ONNXRuntimeError] : 3 : NO_SUCHFILE : Load model from models/xgb_edge.onnx failed:Load model models/xgb_edge.onnx failed. File doesn't exist

In [2]:
import os
# Find where your actual dataset files are
for root, dirs, files in os.walk(os.path.expanduser("~")):
    dirs[:] = [d for d in dirs if d not in ['.git','node_modules','.cache','Library','Applications']]
    for f in files:
        if any(f.endswith(ext) for ext in ['.parquet','.csv','.h5']) and \
           any(kw in f.lower() for kw in ['ton','cicids','unsw','bot','iot']):
            print(os.path.join(root, f))


/Users/malharfalke/College/Btech_Project/Datasets/UNSWNB15/UNSW_NB15_testing-set.parquet
/Users/malharfalke/College/Btech_Project/Datasets/UNSWNB15/UNSW_NB15_training-set.parquet


In [3]:
import os
os.chdir(os.path.expanduser("~/College/Btech_Project"))
print(os.getcwd())



/Users/malharfalke/College/Btech_Project


In [4]:
import onnxruntime as rt
import numpy as np, pandas as pd, joblib, time, json, os

MODELS = "models/"

# Verify files exist
for f in ['xgb_edge.onnx', 'xgb_edge.pkl', 'xgb_unified_4dataset.pkl', 'scaler_unified_4dataset.pkl']:
    path = os.path.join(MODELS, f)
    size = os.path.getsize(path) // 1024
    print(f"✓ {f}  ({size} KB)")

sess    = rt.InferenceSession(os.path.join(MODELS, "xgb_edge.onnx"))
scaler  = joblib.load(os.path.join(MODELS, "scaler_unified_4dataset.pkl"))
in_name = sess.get_inputs()[0].name
print(f"\nONNX input: '{in_name}', shape: {sess.get_inputs()[0].shape}")


✓ xgb_edge.onnx  (45 KB)
✓ xgb_edge.pkl  (37 KB)
✓ xgb_unified_4dataset.pkl  (404 KB)
✓ scaler_unified_4dataset.pkl  (0 KB)

ONNX input: 'input', shape: [None, 12]


In [5]:
# Rebuild scaler from the existing model's training data
# Since ONNX model expects raw input, we need to check if it already includes scaling
# First, test inference WITHOUT scaler to see if it works

import numpy as np

# Try a dummy inference with raw zeros
dummy = np.zeros((1, 12), dtype=np.float32)
out = sess.run(None, {'input': dummy})
print("Raw inference works:", out)

# Check what the scaler file actually contains
import os, joblib
path = 'models/scaler_unified_4dataset.pkl'
print("File size (bytes):", os.path.getsize(path))
try:
    s = joblib.load(path)
    print("Scaler type:", type(s))
    print("Scaler mean:", s.mean_)
except Exception as e:
    print("Scaler load error:", e)


Raw inference works: [array([0], dtype=int64), array([[0.8571285, 0.1428715]], dtype=float32)]
File size (bytes): 903
Scaler type: <class 'sklearn.preprocessing._data.StandardScaler'>
Scaler mean: [1.26962683e+07 1.01969944e+01 1.06181463e+01 1.76658176e+04
 3.15715109e+04 6.79624421e+04 5.57606352e+06 5.75564328e+04
 2.49896216e+03 1.04935620e+03 5.71243274e+02 3.08078578e+02]


In [6]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Load your test slice from notebook 02 — rebuild it here
# Replace these with your actual 12 SFAF feature names from notebook 02
SFAF_FEATURES = list(scaler.feature_names_in_) if hasattr(scaler, 'feature_names_in_') else None
print("Scaler features:", SFAF_FEATURES)
print("Scaler n_features:", scaler.n_features_in_)


Scaler features: None
Scaler n_features: 12


In [8]:
# Recover feature names from the scaler mean values — match against your notebook 02
# The scaler mean shows: [1.26e7, 10.2, 10.6, 1.76e4, 3.15e4, 6.79e4, 5.57e6, 5.75e4, 2.49e3, 1.04e3, 5.71e2, 3.08e2]
# These look like network flow features — check what features were used in notebook 02

import subprocess
result = subprocess.run(['grep', '-n', 'SFAF_FEATURES\|sfaf_features\|feature_cols\|selected_features', 
                        'code/02_SFAF_Unified_Model.ipynb'], 
                       capture_output=True, text=True)
print(result.stdout[:3000])


In [9]:
import json

with open('code/02_SFAF_Unified_Model.ipynb', 'r') as f:
    nb = json.load(f)

# Search all cells for feature name definitions
for i, cell in enumerate(nb['cells']):
    src = ''.join(cell['source'])
    if any(kw in src for kw in ['FEATURES', 'feature_cols', 'columns', 'scaler.fit']):
        print(f"--- Cell {i} ---")
        print(src[:800])
        print()


--- Cell 3 ---
import glob
files = sorted(glob.glob(os.path.join(CICIDS,'*.csv')))
dfs = []
for f in files:
    t = pd.read_csv(f, low_memory=False); t.columns = t.columns.str.strip(); dfs.append(t)
df_cic = pd.concat(dfs, ignore_index=True)
df_cic['label'] = (df_cic['Label'].str.strip().str.upper() != 'BENIGN').astype(int)
print(f"CICIDS2017: {df_cic.shape[0]:,} rows | Attack ratio: {df_cic['label'].mean()*100:.1f}%")

--- Cell 4 ---
df_unsw = pd.concat([
    pd.read_parquet(os.path.join(UNSW,'UNSW_NB15_training-set.parquet')),
    pd.read_parquet(os.path.join(UNSW,'UNSW_NB15_testing-set.parquet'))
], ignore_index=True)
df_unsw.columns = df_unsw.columns.str.strip()
df_unsw['label'] = df_unsw['label'].astype(int)
print(f"UNSW-NB15: {df_unsw.shape[0]:,} rows | Attack ratio: {df_unsw['label'].mean()*100:.1f}%")

--- Cell 5 ---
df_ton = pd.read_csv(os.path.join(TON,'train_test_network.csv'), low_memory=False)
df_ton.columns = df_ton.columns.str.strip()
df_ton['label'] = df_ton['label'].as

In [10]:
import numpy as np, time

UNIFIED_FEATURES = [
    'Flow Duration','Total Fwd Packets','Total Backward Packets',
    'Total Length of Fwd Packets','Total Length of Bwd Packets',
    'Flow Packets/s','Fwd Packets/s','Bwd Packets/s',
    'Min Packet Length','Max Packet Length','Packet Length Mean','Packet Length Std'
]

# Build a synthetic test batch using scaler mean/std (no raw data needed)
rng   = np.random.RandomState(42)
X_syn = rng.normal(loc=scaler.mean_, scale=np.sqrt(scaler.var_), size=(10000,12)).astype(np.float32)
X_scaled = scaler.transform(X_syn).astype(np.float32)

# Latency benchmark across batch sizes
BATCH_SIZES = [1, 8, 32, 64, 128, 512, 1024]
results = []
for bs in BATCH_SIZES:
    batch = X_scaled[:bs]
    times = [((lambda t0: (sess.run(None,{'input':batch}), time.perf_counter()-t0)[1])(time.perf_counter())) for _ in range(300)]
    times = times[20:]  # drop warmup
    results.append({
        'Batch': bs,
        'Mean_ms': round(np.mean(times)*1000, 3),
        'P99_ms':  round(np.percentile(times,99)*1000, 3),
        'Throughput_rps': round(bs / np.mean(times), 1)
    })

import pandas as pd
df_lat = pd.DataFrame(results)
df_lat.to_csv('models/latency_by_batch.csv', index=False)
print(df_lat.to_string(index=False))


 Batch  Mean_ms  P99_ms  Throughput_rps
     1    0.010   0.011        103659.1
     8    0.014   0.020        557091.4
    32    0.024   0.037       1338987.8
    64    0.025   0.031       2555137.1
   128    0.053   0.124       2437603.1
   512    0.168   0.212       3045426.3
  1024    0.331   0.409       3090504.7


In [11]:
import joblib, json

# Model size comparison
sizes = {
    'xgb_unified_4dataset.pkl': os.path.getsize('models/xgb_unified_4dataset.pkl') // 1024,
    'xgb_edge.pkl':             os.path.getsize('models/xgb_edge.pkl') // 1024,
    'xgb_edge.onnx':            os.path.getsize('models/xgb_edge.onnx') // 1024,
}
df_size = pd.DataFrame(sizes.items(), columns=['Model', 'Size_KB'])
df_size['Reduction_pct'] = round((1 - df_size['Size_KB'] / sizes['xgb_unified_4dataset.pkl']) * 100, 1)
df_size.to_csv('models/model_size_comparison.csv', index=False)
print(df_size.to_string(index=False))

# Save benchmark summary as JSON for report
summary = {
    'single_flow_latency_ms': 0.010,
    'single_flow_p99_ms': 0.011,
    'single_flow_throughput_rps': 103659,
    'batch64_latency_ms': 0.025,
    'batch64_throughput_rps': 2555137,
    'onnx_size_kb': sizes['xgb_edge.onnx'],
    'full_model_size_kb': sizes['xgb_unified_4dataset.pkl'],
    'size_reduction_pct': round((1 - sizes['xgb_edge.onnx'] / sizes['xgb_unified_4dataset.pkl']) * 100, 1)
}
with open('models/edge_benchmark.json', 'w') as f:
    json.dump(summary, f, indent=2)
print("\nBenchmark summary:")
print(json.dumps(summary, indent=2))


                   Model  Size_KB  Reduction_pct
xgb_unified_4dataset.pkl      404            0.0
            xgb_edge.pkl       37           90.8
           xgb_edge.onnx       45           88.9

Benchmark summary:
{
  "single_flow_latency_ms": 0.01,
  "single_flow_p99_ms": 0.011,
  "single_flow_throughput_rps": 103659,
  "batch64_latency_ms": 0.025,
  "batch64_throughput_rps": 2555137,
  "onnx_size_kb": 45,
  "full_model_size_kb": 404,
  "size_reduction_pct": 88.9
}
